## MLlib Regression Experiment with Caching

### University of Virginia
### DS 7200: Distributed Computing
### Last Updated: September 22, 2026

---  


#### INSTRUCTIONS: 

This notebook demonstrates construction of a data pipeline and fitting a linear regression model. It illustrates the runtime difference with and without training data caching.

Carefully review the code below, fill in the missing sections, run the code, and note the results

---

In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
import time

# =========================================================
# 0 | SET CONFIGS

LABEL_COLUMN = "median_house_value"
FILENAME_DATA = "california_housing_spark_10000.csv"
MAX_ITER = 20

# =========================================================
# 1 | START SPARK

spark = (
    SparkSession.builder
    .appName("CachingDemo")
    .getOrCreate()
)


# =========================================================
# 2 | READ DATA

#[INSERT CODE TO READ THE DATA FROM THE CSV FILE NAMED ABOVE. IT CAN BE FOUND AT THIS PATH: 
#  /standard/ds7200-apt4c/large_datasets/california_housing_spark_10000.csv]

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"/standard/ds7200-apt4c/large_datasets/{FILENAME_DATA}")
)

print("Number of records:", df.count())

df.printSchema()


# =========================================================
# 3 | FEATURES

features = [
    "median_income",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "latitude",
    "longitude"
]


# =========================================================
# 4 | FEATURE ENGINEERING

#[INSERT CODE TO INSTANTIATE A VECTOR ASSEMBLER TAKING FEATURES FROM ABOVE]
# The vector assembler object should be named: assembler

assembler = VectorAssembler(
    inputCols=features,
    outputCol="raw_features"
)

#[INSERT CODE TO APPLY STANDARD SCALER TO THE FEATURE COLUMN. SCALE THE DATA BUT DON'T SUBTRACT THE MEAN]
# The scaler object should be named: scaler

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=False
)

# A pipeline is used to preprocess the data. This eliminates multiple calls to fit(), simplifies the process,
# and it's reusable.

pipeline = Pipeline(
    stages=[
        assembler,
        scaler
    ]
)


# =========================================================
# 5 | CREATE TRAINING DATA

pipeline_model = pipeline.fit(df)

processed = (
    pipeline_model
    .transform(df)
    .select(
        "features",
        "median_house_value"
    )
)

# =========================================================
# 6 | TRAIN / TEST SPLIT

#[INSERT CODE TO RANDOMLY SPLIT THE DATA INTO 80% train / 20% test]
train, test = processed.randomSplit([0.8, 0.2], seed=42)

print("Training records:", train.count())
print("Test records:", test.count())


# =========================================================
# 7 | LINEAR REGRESSION

#[INSERT CODE TO INSTANTIATE A LINEAR REGRESSION MODEL.]
# It should use the features column as predictors, LABEL_COLUMN for the target variable

lr = LinearRegression(
    featuresCol="features",
    labelCol=LABEL_COLUMN,
    maxIter=MAX_ITER
)


Number of records: 10000
root
 |-- median_house_value: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)

Training records: 8079
Test records: 1921


In [3]:
# =========================================================
# 8 | WITHOUT CACHE

print("\n========================================")
print("WITHOUT CACHE")
print("========================================")

start = time.time()

for i in range(5):

    model = lr.fit(train)

    # Force Spark to perform another action (inference) using the training data.
    predictions = model.transform(train)

    rmse = (
        predictions
        .selectExpr(
            "sqrt(avg(pow(median_house_value - prediction, 2))) as rmse"
        )
        .collect()[0]["rmse"]
    )

    print(
        f"Iteration {i + 1}: "
        f"training RMSE = {rmse:.2f}"
    )

no_cache_time = time.time() - start

print("\nTime without cache:", no_cache_time)


WITHOUT CACHE


26/09/23 16:04:44 WARN Instrumentation: [58c21488] regParam is zero, which might cause numerical instability and overfitting.
26/09/23 16:04:44 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/09/23 16:04:44 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


Iteration 1: training RMSE = 57768.96


26/09/23 16:04:45 WARN Instrumentation: [e6d0540c] regParam is zero, which might cause numerical instability and overfitting.


Iteration 2: training RMSE = 57768.96


...

Iteration 3: training RMSE = 57768.96


26/09/23 16:04:47 WARN Instrumentation: [95cc06b7] regParam is zero, which might cause numerical instability and overfitting.


Iteration 4: training RMSE = 57768.96


26/09/23 16:04:48 WARN Instrumentation: [96ef22e9] regParam is zero, which might cause numerical instability and overfitting.


Iteration 5: training RMSE = 57768.96

Time without cache: 4.594736337661743


In [4]:

# =========================================================
# 9 | WITH CACHE

print("\n========================================")
print("WITH CACHE")
print("========================================")

train_cached = train.cache()

# IMPORTANT:
# Materialize the cache before starting the timer. This is done by prompting an action.
train_cached.count()

start = time.time()

for i in range(5):

    # fit the model using the cached training data
    model = lr.fit(train_cached)

    predictions = model.transform(train_cached)

    rmse = (
        predictions
        .selectExpr(
            "sqrt(avg(pow(median_house_value - prediction, 2))) as rmse"
        )
        .collect()[0]["rmse"]
    )

    print(
        f"Iteration {i + 1}: "
        f"training RMSE = {rmse:.2f}"
    )

cache_time = time.time() - start

print("\nTime with cache:", cache_time)


# =========================================================
# 10 | COMPARISON

print("\n========================================")
print("RESULTS")
print("========================================")

print(f"Without cache: {no_cache_time:.2f} seconds")
print(f"With cache:    {cache_time:.2f} seconds")

if cache_time > 0:
    speedup = no_cache_time / cache_time
    print(f"Speedup:       {speedup:.2f}x")


# =========================================================
# CLEAN UP
# =========================================================

train_cached.unpersist()

spark.stop()


WITH CACHE


26/09/23 16:06:17 WARN Instrumentation: [2074c84c] regParam is zero, which might cause numerical instability and overfitting.


Iteration 1: training RMSE = 57768.96


26/09/23 16:06:17 WARN Instrumentation: [9ac52182] regParam is zero, which might cause numerical instability and overfitting.


Iteration 2: training RMSE = 57768.96


...

Iteration 3: training RMSE = 57768.96


26/09/23 16:06:18 WARN Instrumentation: [708eb749] regParam is zero, which might cause numerical instability and overfitting.


Iteration 4: training RMSE = 57768.96


26/09/23 16:06:18 WARN Instrumentation: [fd28fa18] regParam is zero, which might cause numerical instability and overfitting.


Iteration 5: training RMSE = 57768.96

Time with cache: 1.7517132759094238

RESULTS
Without cache: 4.59 seconds
With cache:    1.75 seconds
Speedup:       2.62x
